# 전처리 연습 (Tokenize, Cleansing & Normalization, Stemming & Lemmatization)

1. 데이터셋(corpus)을 찾는다. (만들어진 데이터셋, 크롤링, ...)
    - **🐿️ 이어지는 실습에 쓸 수 있을 법한 데이터 찾기 Tip**
    - 여러 문장으로 이루어진 데이터셋이라면 일단 good
        - 리뷰 등 이어지지 않는 짧은 문장 여러 건 ok
        - 소설 등 이어지는 긴 문장 여러 건 ok
    - 대화형 데이터셋도 good
        - QnA로 구성되어 있는 corpus도 좋고
        - 일반적인 대화로 구성되어 있는 corpus도 좋아요~
    - 특정 도메인에 대한 정보를 담고 있는 데이터셋도 good
2. 전처리를 해본다.
    - 텍스트 자체를 깔끔하게 만드는 것까지만

In [1]:
import os, glob
import pandas as pd

path = 'data/raw/articles'
all_files = glob.glob(os.path.join(path, '*.csv'))
df = pd.DataFrame()

for temp_file in all_files:
    temp_df = pd.read_csv(temp_file, encoding='utf-8')
    df = pd.concat([df, temp_df])

df.to_csv('data/preprocessed/total_articles.csv')

In [2]:
df['content'] = df['content'].replace(r'\[.*?\]', '', regex=True)
df['content'] = df['content'].replace(r'\<.*?\>', '', regex=True)
df['content'] = df['content'].replace(r'ⓒ.*?\)', '', regex=True)
df['content'] = df['content'].replace(r'[\r\n]', '', regex=True)
df['content'] = df['content'].replace(r'사진=', '', regex=True)
df['content'] = df['content'].replace(r'\(.*?기자\)', '', regex=True)
df['content'] = df['content'].replace(r'[a-zA-Z0-9]+@[a-zA-Z0-9]+\.[a-zA-Z0-9]', '', regex=True)
df['content'] = df['content'].replace(r'[^0-9ㄱ-ㅎ가-힣ㅏ-ㅣ\.\?\! ]', '', regex=True)

In [3]:
def load_ko_stopwords(filepath):
    with open(filepath, 'r', encoding='UTF-8') as f:
        return [line.strip() for line in f]
    
ko_stopwords = load_ko_stopwords('./data/ko_stopwords.txt')

In [4]:
# from tqdm import tqdm
# import kss

# corpus = [kss.split_sentences(content) for content in tqdm(df['content'])]
# print(corpus)

In [5]:
from tqdm import tqdm
from nltk.tokenize import sent_tokenize

corpus = [sent_tokenize(content) for content in tqdm(df['content'])]

100%|██████████| 14575/14575 [00:03<00:00, 4207.52it/s]


In [6]:
from tqdm import tqdm
from konlpy.tag import Okt

okt = Okt()

preprocessed_data = []

for sentences in tqdm(corpus):
    preprocessed_sentences = []

    for sentence in sentences:
        tokens = okt.morphs(sentence, stem=True)
        tokens = [token for token in tokens if token not in ko_stopwords]
        preprocessed_sentences.append(tokens)
        
    preprocessed_data.append(preprocessed_sentences)

100%|██████████| 14575/14575 [15:24<00:00, 15.77it/s]


In [7]:
reconstructed_corpus = []

for article in preprocessed_data:
    reconstructed_article = "\n".join([" ".join(sentence) for sentence in article])
    reconstructed_corpus.append(reconstructed_article)

df_processed = pd.DataFrame({'reconstructed_content': reconstructed_corpus})
df_processed.to_csv('./data/preprocessed/final_preprocessed_text.csv', index=False, encoding='utf-8')